# Plot the results from logs/

Reads all CSVs in `logs/` (handles both the IQ-Learn and CSIL naming conventions) and produces learning curves and sample-efficiency plots to `plots/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
%cd /content/drive/MyDrive/imitation_learning

In [ ]:
import os, csv
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt

os.makedirs('plots', exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'font.size': 11, 'axes.titlesize': 13, 'axes.labelsize': 11,
    'axes.grid': True, 'grid.alpha': 0.3, 'lines.linewidth': 2.0,
})

ALGOS = ['iqlearn', 'csil', 'csilsoar']
ALGO_LABEL = {'iqlearn': 'IQ-Learn', 'csil': 'CSIL', 'csilsoar': 'CSIL+SOAR'}
ALGO_COLOR = {'iqlearn': '#1f77b4', 'csil': '#ff7f0e', 'csilsoar': '#2ca02c'}
ENVS = ['CartPole', 'Pendulum']
K_VALUES = [1, 3, 5, 10, 15]
SEEDS = [42, 43, 44]

In [ ]:
def load_one(path):
    steps, rewards = [], []
    with open(path) as f:
        for row in csv.DictReader(f):
            steps.append(int(row['step']))
            rewards.append(float(row['eval_reward']))
    return np.array(steps), np.array(rewards)

def candidate_paths(algo, env, K, seed):
    return [f'logs/{algo}_{env}{s}_K{K}_seed{seed}.csv' for s in ['', '-v1']]

data = defaultdict(lambda: defaultdict(lambda: defaultdict(dict)))
for algo in ALGOS:
    for env in ENVS:
        for K in K_VALUES:
            for seed in SEEDS:
                for p in candidate_paths(algo, env, K, seed):
                    if os.path.exists(p):
                        data[algo][env][K][seed] = load_one(p)
                        break

for algo in ALGOS:
    for env in ENVS:
        n = sum(len(data[algo][env][K]) for K in K_VALUES)
        print(f'  {algo:10s} {env:10s}: {n}/15')

In [ ]:
def aggregate(seed_dict):
    if not seed_dict:
        return None
    L = min(len(v[0]) for v in seed_dict.values())
    steps = list(seed_dict.values())[0][0][:L]
    rewards = np.array([v[1][:L] for v in seed_dict.values()])
    return steps, np.median(rewards, axis=0), np.percentile(rewards, 25, axis=0), np.percentile(rewards, 75, axis=0)

In [ ]:
K_COLORS = plt.cm.viridis(np.linspace(0.15, 0.85, len(K_VALUES)))

def plot_learning_curves(algo, env, ax=None, save=True):
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 4.5))
    for K, color in zip(K_VALUES, K_COLORS):
        agg = aggregate(data[algo][env][K])
        if agg is None:
            continue
        steps, median, p25, p75 = agg
        ax.plot(steps, median, color=color, label=f'K={K}', linewidth=2)
        ax.fill_between(steps, p25, p75, color=color, alpha=0.18)
    ax.set_xlabel('Training steps')
    ax.set_ylabel('Evaluation reward (median over 3 seeds)')
    ax.set_title(f'{ALGO_LABEL[algo]} on {env}-v1')
    ax.legend(loc='best', frameon=True, framealpha=0.9)
    if save:
        plt.gcf().savefig(f'plots/learning_curves_{algo}_{env}.png')

for algo in ALGOS:
    for env in ENVS:
        if not data[algo][env]:
            continue
        plt.figure()
        plot_learning_curves(algo, env)
        plt.show()

In [ ]:
def max_reward_per_seed(seed_dict):
    return [v[1].max() for v in seed_dict.values()]

def plot_sample_efficiency(env, ax=None, save=True):
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 4.5))
    for algo in ALGOS:
        xs, med, p25, p75 = [], [], [], []
        for K in K_VALUES:
            maxes = max_reward_per_seed(data[algo][env][K])
            if not maxes:
                continue
            xs.append(K)
            med.append(np.median(maxes))
            p25.append(np.percentile(maxes, 25))
            p75.append(np.percentile(maxes, 75))
        if not xs:
            continue
        med, p25, p75 = map(np.array, [med, p25, p75])
        ax.errorbar(xs, med, yerr=[med - p25, p75 - med],
                    color=ALGO_COLOR[algo], label=ALGO_LABEL[algo],
                    marker='o', capsize=4, linewidth=2)
    ax.set_xlabel('K (number of expert trajectories)')
    ax.set_ylabel('Best evaluation reward (median over 3 seeds)')
    ax.set_title(f'Sample efficiency on {env}-v1')
    ax.set_xscale('log')
    ax.set_xticks(K_VALUES)
    ax.set_xticklabels(K_VALUES)
    ax.legend(loc='best', frameon=True, framealpha=0.9)
    if save:
        plt.gcf().savefig(f'plots/sample_efficiency_{env}.png')

for env in ENVS:
    plt.figure()
    plot_sample_efficiency(env)
    plt.show()